<a href="https://colab.research.google.com/github/angelpavu/A03-knn/blob/main/notebook_(4).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Assignment 3: $k$ Nearest Neighbor


**Q1.**
1. What is the difference between regression and classification?
2. What is a confusion table? What does it help us understand about a model's performance?
3. What does the SSE quantify about a particular model?
4. What are overfitting and underfitting?
5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?
6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.

**Q2.** This is a case study on $k$ nearest neighbor classification, using the `land_mines.csv` data.

The data consists of a label, `mine_type`, taking integer values 1 to 5, and three properties of the mine, `voltage`, `height` and `soil`. We want to predict the kind of mine from data about it. Imagine working for the DOD or a humanitarian aid agency, trying to help people remove land mines more safely.

1. Load the data. Perform some EDA, summarizing the target label and the features.
2. Split the sample 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be, in my experience: Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)
3. Build a $k$-NN classifier. Explain how you select $k$.
4. Print a confusion table for the optimal model, comparing predicted and actual class label on the test set. How accurate is it? Where is performance more or less accurate?
5. Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** Angel Pavu

**Date:** 10/01/2026

## Question 1
1. Regression predicts a numeric outcome, such as baseline sales. Regression error is measured by how far the prediction is from the true number. Classification predicts a category, such as class. Error in classification is measured by whether the label was right or wrong.

2. A confusion table counts predictions by actual class and predicted class. It has four cells: true positives, true negatives, false positives and false negatives. It helps because from the table you can compute metrics like precision, recall, and specificity, and see whether errors are concentrated in one class. It also shows which kinds of mistakes are costly (a false negative on a cancer screen is usually worse than a false positive).

3. SSE (sum of squared error) = Σ(yᵢ − ŷᵢ)², the sum over observations of the squared difference between actual and predicted values. It quantifies the total amount of error a regression model makes. Lower validation SSE means smaller errors on this validation set.

4. Underfitting is when the model is too simple to capture the real pattern in the data (for example fitting a straight line to a curved relationship, or using a very large k in kNN). It performs poorly on both training and test data and is high in bias.
Overfitting is when the model is too flexible and learns the noise and quirks of the training sample rather than the underlying pattern. It performs very well on training data but poorly on new data and is high variance.

5. Error on the training data is misleading, because a model can always look better there by becoming more complex. Holding out a test set gives an honest estimate of how the model performs on data it hasn't seen, which is what we actually care about. Evaluating accuracy or SSE on the test set across different values of k lets us see where test error is lowest. Small k tends to overfit, large k tends to underfit, and the test curve reveals the best trade-off. So the split doesn't improve the model by itself but rather it lets us select a model that generalizes better and gives a realistic estimate of future performance.

6. Class labels are simple, easy to communicate, and directly actionable, but they hide how confident the model is (a 51% prediction looks the same as a 99% one) and lock in a fixed threshold. Probabilities convey confidence, let you adjust the threshold to match the costs of different errors, and allow you to rank cases, but they are harder to interpret and can be poorly calibrated, so an "80%" prediction may not really be right 80% of the time.


In [1]:
# Question 2
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, classification_report

df = pd.read_csv("land_mines.csv")
print(df.shape, df.isna().sum().sum())
print(df.mine_type.value_counts().sort_index())
print(df.groupby("mine_type")[["voltage","height","soil"]].agg(["mean","std"]))



(338, 4) 0
mine_type
1    71
2    70
3    66
4    66
5    65
Name: count, dtype: int64
            voltage              height                soil          
               mean       std      mean       std      mean       std
mine_type                                                            
1          0.296463  0.031672  0.495519  0.315962  0.492958  0.341143
2          0.721123  0.222267  0.487013  0.310720  0.508571  0.345045
3          0.402408  0.098185  0.527548  0.296288  0.496970  0.351248
4          0.345365  0.092564  0.506887  0.306348  0.503030  0.347727
5          0.379598  0.076340  0.530070  0.306426  0.516923  0.346216


The target, mine_type, has 5 nearly balanced classes (65 to 71 mines each, 338 total), so chance accuracy is about 20%. The three features are already scaled to roughly 0 to 1 and voltage is the only one that separates the classes (type 2 averages 0.72 versus 0.30 to 0.40 for the rest), while height and soil look nearly identical across all types.

In [2]:
# train/test split
X, y = df[["voltage","height","soil"]], df.mine_type
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.5, stratify=y, random_state=0)

In [5]:
# Build a  k -NN classifier.
def knn(k):
    return make_pipeline(MinMaxScaler(), KNeighborsClassifier(k))

acc = {k: knn(k).fit(Xtr, ytr).score(Xte, yte) for k in range(1, 41)}
best_k = max(acc, key=acc.get)
print("Best k:", best_k, "| Test accuracy:", round(acc[best_k], 3))

Best k: 2 | Test accuracy: 0.42


I fit the k-NN model for every k from 1 to 40 on the training half and compared accuracy on the held-out test half, then picked the k with the highest accuracy, which was k = 2 (about 42%). Small k overfits (k = 1 was only 38%) and large k underfits (accuracy fell to about 31 to 35% by k = 30 to 40), so the best k sits at a low value in between. Cross-validation on the training half also picked k = 2, which supports the choice.

In [6]:
# Print a confusion table for the optimal model
model = knn(best_k).fit(Xtr, ytr)
pred = model.predict(Xte)
print(pd.crosstab(yte, pred, rownames=["Actual"], colnames=["Predicted"]))

Predicted   1   2   3  4  5
Actual                     
1          23   0  11  2  0
2           0  28   1  3  3
3          10   3  12  3  5
4          14   2  10  7  0
5          12   0  15  4  1


The optimal model (k = 2) classifies 71 of the 169 test mines correctly, for an overall accuracy of about 42%, which is well above the 20% you would get by guessing among five roughly balanced classes. Performance is strongest for mine type 2, where the model correctly identifies 28 of 35 mines (80% recall), and 85% of its "type 2" predictions are correct, which makes sense because type 2 has a distinctly high voltage. Type 1 is found fairly often (23 of 36, or 64%), but its precision is only 39%, because many type 3, 4 and 5 mines are mislabeled as type 1. Performance is weakest for types 3, 4 and 5, which have heavily overlapping voltage values and get confused with one another and with type 1. Type 3 is correct only 36% of the time, type 4 only 21%, and type 5 is almost never identified correctly (1 of 32, or 3%).

### Part 5
The model's accuracy is uneven and no mine should ever be handled based on a predicted label alone. The only prediction worth acting on is "type 2" (about 85% precision), while predictions of types 1, 3, 4 and 5 are wrong often enough that those mines should be handled under the most cautious protocol until they are physically identified. It would also help to report the neighbor-vote probabilities instead of just labels and act only on high-confidence predictions, sending low-confidence cases to manual identification. Finally, because misidentifying some mine types is far more dangerous than others, decisions should be based on the cost of errors, not just the most likely class, and the real fix is collecting more informative measurements, since only voltage currently separates the classes.

## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** [To be provided]
- **AI tool and model used:** [To be provided]

### *Prompts used

1. [Paste prompt]
2. [Paste follow-up prompt]

### *AI-proposed changes


[Paste the AI's concise numbered list verbatim.]

1. ...
2. ...

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | Accept / modify / reject | ... |
| 2 | Accept / modify / reject | ... |

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

In [ ]:
# Your Phase 2 solution to the problem goes here.


### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]